# 01_data_preprocessing: Sector Integration

## 日本語
このNotebookは、論文再現のための3段階ワークフローの第1部です。

分析で用いる部門分類に合わせて、生の地域間産業連関表を前処理・統合します。

生表は ../data/raw/intermunicipal_io_tables/ から読み込み、対応表ファイルは ../data/mapping/ から読み込みます。

処理後の出力は ../output/tables/ に、S60_integrated.xlsx、H2_integrated.xlsx、H7_integrated.xlsx、H17_integrated.xlsx として保存されます。

想定している入力ファイル名は、2026年4月時点での経産省HPからダウンロードした際のファイル名、もしくは `S60.xlsx`, `H2.xlsx`, `H7.xlsx`, `H17.xlsx` です。

## English
This notebook is the first part of the three-step reproduction workflow for the paper.
It performs the data preprocessing required to harmonize the raw interregional input–output tables into the sector classification used in the analysis.

It reads raw input-output tables from `../data/raw/intermunicipal_io_tables/` and the concordance file from `../data/mapping/`.

Integrated outputs are written to `../output/tables/` as `S60_integrated.xlsx`, `H2_integrated.xlsx`, `H7_integrated.xlsx`, and `H17_integrated.xlsx`, and can be used in subsequent notebooks.

The expected input filenames are either the filenames obtained when downloading the files from the METI website as of April 2026, `S60.xlsx`, `H2.xlsx`, `H7.xlsx`, and `H17.xlsx`.


# Sector Integration / 部門統合


In [ ]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

# このスクリプト or notebook を置いている基準フォルダ
BASE_DIR = Path.cwd()

# 参照先をここで一元管理
MAPPING_DIR = BASE_DIR / "data" / "mapping"
RAW_DIR = BASE_DIR / "data" / "raw" / "intermunicipal_io_tables"
OUTPUT_DIR = BASE_DIR / "output" / "tables"

# =========================
# 追加: 読み取り直後の標準化
# =========================

def normalize_text(x):
    if pd.isna(x):
        return ""
    s = str(x)
    s = s.replace("\t", "")
    s = s.replace("\u3000", " ")   # 全角スペース
    s = s.strip()
    s = re.sub(r"\s+", "", s)
    return s

# 元表の表記ゆれを、後段コードが期待する標準名へ寄せる
CANONICAL_MAP = {
    "一般政府消費支出": "政府消費支出",
    "地域内総固定資本形成（公的）": "地域内総固定資本形成",
    "地域内総固定資本形成（民間）": "地域内総固定資本形成",
    "製品・半製品・仕掛品在庫純増": "在庫純増",
    "流通・原材料在庫純増": "在庫純増",
    "（控除）輸移入計": "（控除）輸入",
    "（控除）輸入計": "（控除）輸入",
    "間接税（除関税・輸入品商品税）": "間接税",
}

REGION_NAMES = {
    "北海道", "東北", "関東", "中部", "近畿",
    "中国", "四国", "九州", "沖縄", "地域計"
}

def canonicalize_label(x):
    s = normalize_text(x)
    return CANONICAL_MAP.get(s, s)

def standardize_io_table(df_raw):
    df = df_raw.copy()

    # 文字列セルだけ標準化する
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            val = df.iat[i, j]
            if isinstance(val, str):
                df.iat[i, j] = canonicalize_label(val)

    # B列（地域名）を整える
    if df.shape[1] > 1:
        for i in range(df.shape[0]):
            val = df.iat[i, 1]
            if isinstance(val, str):
                v = normalize_text(val)
                if v in REGION_NAMES:
                    df.iat[i, 1] = v

    # 4行目（列側地域名）を整える
    if df.shape[0] > 3:
        for j in range(df.shape[1]):
            val = df.iat[3, j]
            if isinstance(val, str):
                v = normalize_text(val)
                if v in REGION_NAMES:
                    df.iat[3, j] = v

    return df

def load_mapping_file(file_path):  # マッピングファイルを読み込み
    try:
        df = pd.read_excel(file_path)
        return df
    except Exception as e:
        print(f"マッピングファイルの読み込みエラー: {e}")
        return None

def create_integration_operations(mapping_df, year_column):
    operations = []
    industries_to_remove = []  # 削除すべき産業リスト
    grouped_data = {}

    # 統合部門ごとに産業をグループ化
    for idx, row in mapping_df.iterrows():
        integrated_sector = row['統合部門']
        industry = row[year_column]

        # 統合部門が空で産業が存在する場合、削除リストに追加
        if pd.isna(integrated_sector) and not pd.isna(industry):
            if isinstance(industry, str):
                inds_to_remove = re.split('、|,', industry)
                for ind in inds_to_remove:
                    ind = ind.strip()
                    if ind and ind not in industries_to_remove:
                        industries_to_remove.append(ind)
            continue

        # 両方とも空の場合はスキップ
        if pd.isna(integrated_sector) or pd.isna(industry):
            continue

        if integrated_sector not in grouped_data:
            grouped_data[integrated_sector] = []

        if isinstance(industry, str):
            industries = re.split('、|,', industry)
            for ind in industries:
                ind = ind.strip()
                if ind and ind not in grouped_data[integrated_sector]:
                    grouped_data[integrated_sector].append(ind)

    for integrated_sector, industries in grouped_data.items():
        operations.append({
            "industries_to_integrate": industries,
            "new_industry_name": integrated_sector
        })

    return operations, industries_to_remove

def remove_industries_row(df, region, industries_to_remove, region_col_idx=1, industry_col_idx=3, data_start_row=6):
    """行方向で不要な産業を削除"""
    result_df = df.copy()
    rows_to_drop = []

    for i in range(data_start_row, len(result_df)):
        if i >= len(result_df) or region_col_idx >= len(result_df.columns) or industry_col_idx >= len(result_df.columns):
            continue

        current_region = str(result_df.iloc[i, region_col_idx]) if pd.notna(result_df.iloc[i, region_col_idx]) else ""
        current_industry = str(result_df.iloc[i, industry_col_idx]) if pd.notna(result_df.iloc[i, industry_col_idx]) else ""

        if current_region == region and current_industry in industries_to_remove:
            rows_to_drop.append(i)

    if rows_to_drop:
        result_df = result_df.drop(index=rows_to_drop).reset_index(drop=True)

    return result_df

def remove_industries_column(df, region, industries_to_remove, region_row_idx=3, industry_row_idx=5, data_start_col=4):
    """列方向で不要な産業を削除"""
    result_df = df.copy()
    cols_to_drop = []

    for i in range(data_start_col, len(result_df.columns)):
        if (i < len(result_df.columns) and
            region_row_idx < len(result_df) and
            industry_row_idx < len(result_df)):

            region_val = str(result_df.iloc[region_row_idx, i]) if pd.notna(result_df.iloc[region_row_idx, i]) else ""
            industry_val = str(result_df.iloc[industry_row_idx, i]) if pd.notna(result_df.iloc[industry_row_idx, i]) else ""

            if region_val == region and industry_val in industries_to_remove:
                cols_to_drop.append(i)

    if cols_to_drop:
        cols_to_keep = [i for i in range(len(result_df.columns)) if i not in cols_to_drop]
        result_df = result_df.iloc[:, cols_to_keep]

    return result_df

def integrate_industries_row(df, region, integration_operations, region_col_idx=1, industry_col_idx=3, data_start_row=6):
    """行方向で産業を統合（1対1の名称変更も含む）"""
    result_df = df.copy()

    for operation in integration_operations:
        industries_to_integrate = operation["industries_to_integrate"]
        new_industry_name = operation["new_industry_name"]

        target_rows = []
        for i in range(data_start_row, len(result_df)):
            if i >= len(result_df) or region_col_idx >= len(result_df.columns) or industry_col_idx >= len(result_df.columns):
                continue

            current_region = str(result_df.iloc[i, region_col_idx]) if pd.notna(result_df.iloc[i, region_col_idx]) else ""
            current_industry = str(result_df.iloc[i, industry_col_idx]) if pd.notna(result_df.iloc[i, industry_col_idx]) else ""

            if current_region == region and current_industry in industries_to_integrate:
                target_rows.append(i)

        if not target_rows:
            continue

        insert_row = min(target_rows)
        result_df.iloc[insert_row, industry_col_idx] = new_industry_name

        for col in range(4, len(result_df.columns)):
            sum_value = 0
            for row in target_rows:
                if row < len(result_df) and col < len(result_df.columns):
                    val = result_df.iloc[row, col]
                    if isinstance(val, (int, float)) and not pd.isna(val):
                        sum_value += val

            if insert_row < len(result_df) and col < len(result_df.columns):
                result_df.iloc[insert_row, col] = sum_value

        rows_to_drop = target_rows[1:]
        if rows_to_drop:
            result_df = result_df.drop(index=rows_to_drop).reset_index(drop=True)

    return result_df

def integrate_industries_column(df, region, integration_operations, region_row_idx=3, industry_row_idx=5, data_start_col=4):
    """列方向で産業を統合（1対1の名称変更も含む）"""
    result_df = df.copy()

    for operation in integration_operations:
        industries_to_integrate = operation["industries_to_integrate"]
        new_industry_name = operation["new_industry_name"]

        target_cols = []
        for i in range(data_start_col, len(result_df.columns)):
            if (i < len(result_df.columns) and
                region_row_idx < len(result_df) and
                industry_row_idx < len(result_df)):

                region_val = str(result_df.iloc[region_row_idx, i]) if pd.notna(result_df.iloc[region_row_idx, i]) else ""
                industry_val = str(result_df.iloc[industry_row_idx, i]) if pd.notna(result_df.iloc[industry_row_idx, i]) else ""

                if region_val == region and industry_val in industries_to_integrate:
                    target_cols.append(i)

        if not target_cols:
            continue

        insert_col = min(target_cols)

        if industry_row_idx < len(result_df) and insert_col < len(result_df.columns):
            result_df.iloc[industry_row_idx, insert_col] = new_industry_name

        for row in range(6, len(result_df)):
            if row >= len(result_df):
                continue

            sum_value = 0
            for col in target_cols:
                if col < len(result_df.columns):
                    val = result_df.iloc[row, col]
                    if isinstance(val, (int, float)) and not pd.isna(val):
                        sum_value += val

            if insert_col < len(result_df.columns):
                result_df.iloc[row, insert_col] = sum_value

        cols_to_drop = target_cols[1:]
        if cols_to_drop:
            cols_to_keep = [i for i in range(len(result_df.columns)) if i not in cols_to_drop]
            result_df = result_df.iloc[:, cols_to_keep]

    return result_df

def process_io_table(file_path, integration_operations, industries_to_remove):
    """与えられた統合操作と削除対象産業で産業連関表ファイルを処理"""
    try:
        # 最初のシートを読み込み → 標準化
        df_raw = pd.read_excel(file_path, header=None)
        df = standardize_io_table(df_raw)
        print(f"{file_path} の最初のシートを読み込み、標準化しました")
    except Exception as e:
        print(f"{file_path} の読み込みエラー: {e}")
        return None

    regions = ['北海道', '東北', '関東', '中部', '近畿', '中国', '四国', '九州', '沖縄', '地域計']

    # まず不要な産業を削除
    for region in regions:
        df = remove_industries_row(df, region, industries_to_remove)
        df = remove_industries_column(df, region, industries_to_remove)

    # その後に産業の統合を処理
    for region in regions:
        df = integrate_industries_row(df, region, integration_operations)
        df = integrate_industries_column(df, region, integration_operations)

    # 保存前にタブ文字を削除
    df = df.applymap(lambda x: x.replace('\t', '') if isinstance(x, str) else x)

    # B列が「地域計」かつD列が「地域内生産額」の行を探し、それより下の行を削除
    drop_row_index = -1
    b_col_idx = 1
    d_col_idx = 3

    for i in range(len(df)):
        if i < len(df) and b_col_idx < len(df.columns) and d_col_idx < len(df.columns):
            b_val = str(df.iloc[i, b_col_idx]) if pd.notna(df.iloc[i, b_col_idx]) else ""
            d_val = str(df.iloc[i, d_col_idx]) if pd.notna(df.iloc[i, d_col_idx]) else ""

            if b_val == '地域計' and d_val == '地域内生産額':
                drop_row_index = i
                break

    if drop_row_index != -1 and drop_row_index + 1 < len(df):
        df = df.iloc[:drop_row_index + 1].reset_index(drop=True)
        print("💡 B列が'地域計'かつD列が'地域内生産額'の行以下の行を削除しました。")

    return df

def resolve_input_file(raw_dir, candidates):
    """
    raw_dir の中から candidates のファイル名を順に探し、
    見つかったら (Path, ファイル名) を返す。
    見つからなければ (None, None) を返す。
    """
    raw_dir = Path(raw_dir)

    for name in candidates:
        p = raw_dir / name
        if p.exists():
            return p, name

    return None, None

def main():
    # マッピングファイル
    mapping_path = MAPPING_DIR / "産業分類統一対応表_S60_H2_H7_H17.xlsx"
    mapping_df = load_mapping_file(mapping_path)

    if mapping_df is None:
        print("マッピングファイルを読み込めませんでした")
        return

    s60_operations, s60_to_remove = create_integration_operations(mapping_df, '1985年（S60）')
    h2_operations, h2_to_remove = create_integration_operations(mapping_df, '1990年（H2）')
    h7_operations, h7_to_remove = create_integration_operations(mapping_df, '1995年（H7）')
    h17_operations, h17_to_remove = create_integration_operations(mapping_df, '2005年（H17）')

    raw_dir = RAW_DIR
    output_dir = OUTPUT_DIR
    os.makedirs(output_dir, exist_ok=True)

    years = [
        {
            'key': 'S60',
            'candidates': ['S60.xlsx', 'h2rio85a.xlsx'],
            'operations': s60_operations,
            'to_remove': s60_to_remove,
            'output': output_dir / 'S60_integrated.xlsx'
        },
        {
            'key': 'H2',
            'candidates': ['H2.xlsx', 'h2rio90a.xlsx'],
            'operations': h2_operations,
            'to_remove': h2_to_remove,
            'output': output_dir / 'H2_integrated.xlsx'
        },
        {
            'key': 'H7',
            'candidates': ['H7.xlsx', 'h2rio95c.xlsx'],
            'operations': h7_operations,
            'to_remove': h7_to_remove,
            'output': output_dir / 'H7_integrated.xlsx'
        },
        {
            'key': 'H17',
            'candidates': ['H17.xlsx', 'H17年地域間産業連関表(53部門).xlsx'],
            'operations': h17_operations,
            'to_remove': h17_to_remove,
            'output': output_dir / 'H17_integrated.xlsx'
        }
    ]

    for year in years:
        file_path, detected_name = resolve_input_file(raw_dir, year['candidates'])
        print(f"\n{year['key']} を処理中...")

        if file_path is None:
            print(f"未検出: {year['candidates']} が {raw_dir} に見つかりません")
            continue

        print(f"入力ファイル: {detected_name}")
        result = process_io_table(file_path, year['operations'], year['to_remove'])

        if result is not None:
            result.to_excel(year['output'], index=False, header=False)
            print(f"{detected_name} を {year['output']} に保存しました")
        else:
            print(f"{detected_name} の処理に失敗しました")

    print("\n全処理が終了しました")

# メイン関数を実行
if __name__ == "__main__":
    main()